# English STELLA Transcriptions Dataset

The STELLA dataset...


# Data preparation

Format Dataset into the wanted architecture. This procedure extracts audiobook transcriptions from the original dataset and sorts them into the same splits as the audio files.

```
txt
├── LANG
│   ├── HOUR_SPLIT
│   │   ├── SECTION_SPLIT
│   │   │   ├── books.txt
│   │   │   ├── meta.json
│   │   │   └── transcription.txt
│   │   ├── ...
│   ├── ...
│   ...

```
- txt : folder containing transcriptions
- LANG: corresponds to the given language
- HOUR_SPLIR: corresponds to the size of the section splits in number of hours of speech,
              formatted as (50h, 100h, ..., 3200h)
- SECTION_SPLIT: separation of content into sections with equal amount of speech content.
- books.txt: the list of books used for this split
- meta.json: metadata generated during clean-up used to measure effectiveness of cleaning.
- transcript.txt: the agregated transcripts of the audiobooks in the list.

In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

prep = stella.STELAPrepTranscripts(lang="EN")
with timed_status(status="Preping stela transcriptions...", complete_status="Succesfuly build STELA Transcript dataset !"):
    prep.build_transcript()

## Data Cleaning

Clean-up text to keep only clean words that can be piped through the dictionairy.

RULES (Order Matters):
1) Illustration tag removal
2) URL removal
3) TextNormalisation : correct accents & remove non-printable characters
4) Trancribe numbers
5) Remove roman numerals
6) Fix symbols ($,€, etc..)
7) AZFilter

    * replace '-' with a space to extract hyphenated words (fifty-five -> fifty five)

    * Keeps apostrophe char(*'*) to protect shorthands (ex: ain't)
  
    * purges everything not between [A-Z].

    * lowecases everything
8) Fix words by removing prefix and trailing quote char (')

In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import DatasetCleaner, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()
with timed_status(status="Cleaning STELA Transcripts", complete_status="Succesfuly cleanned up STELA Transcript dataset !"):
    DatasetCleaner.cleanup_files(
        filemap=dataset.raw2clean_filesmap("EN"),
        ruleset=dataset.clean_up_rules("EN"),
        save_logs=True
    )

# Word Filtering

Using a pre-selected lexicon we filter the corpus to separated known from unknown words

In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()

with timed_status(status="Word Filtering", complete_status="Succesfuly completed word filtering !"):
    dataset_utils.DatasetCleaner.word_validate_files(
        filemap=dataset.word_validation_filesmap("EN"),
        cleaner=dataset_utils.DictionairyCleaner(lang="EN"),
    )

## Compute word frequency maps

To allow statistics on word cleaning we generate word_frequency table for all steps of the cleaning process :

1) raw transcription frequency maps
2) unvalidated clean transcriptions frequency maps
3) clean transcription frequency maps


In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import DatasetCleaner, DictionairyCleaner, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()

with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    for item in dataset.iter_all():
        _ = item.clean_meta.raw_word_frequencies()
        _ = item.clean_meta.clean_word_frequencies()
        _ = item.clean_meta.rejected_word_frequencies()

## Computing cleanup stats

#### Global Rejection rates

We aggregate word frequency of the raw/clean/rejected accross the whole dataset & compute word rejection rate

In [1]:
import platform

import pandas as pd
from IPython.display import display
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


freq_builder = stella.STELAWordFrequencies()

with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    raw_wf = freq_builder.word_frequencies(word_type="raw")["EN"]
    rejected_wf = freq_builder.word_frequencies(word_type="rejected")["EN"]
    clean_wf = freq_builder.word_frequencies(word_type="clean")["EN"]
    stela_total_wstats_df = pd.DataFrame([
        {"set": "raw", "tokens": raw_wf["freq"].sum(), "types": len(raw_wf["word"])},
        {"set": "rejected", "tokens": rejected_wf["freq"].sum(), "types": len(rejected_wf["word"])},
        {"set": "clean", "tokens": clean_wf["freq"].sum(), "types": len(clean_wf["word"])},
    ])
    stela_total_type_rejection_rate = len(rejected_wf["word"]) / len(raw_wf["word"])
    stela_total_token_rejection_rate = rejected_wf["freq"].sum() / raw_wf["freq"].sum()

%store stela_total_wstats_df
%store stela_total_type_rejection_rate
%store stela_total_token_rejection_rate

Output()

Succesfully computed all word frequencies ! (Total time: 37 seconds)

Stored 'stela_total_wstats_df' (DataFrame)
Stored 'stela_total_type_rejection_rate' (float)
Stored 'stela_total_token_rejection_rate' (float64)


In [3]:
%store -r stela_total_wstats_df
%store -r stela_total_type_rejection_rate
%store -r stela_total_token_rejection_rate
display(
    stela_total_wstats_df.style.format({
        "tokens": "{:,}",
        "types": "{:,}",
    })
)
print(f"""
Token rejection rate across the STELATranscription/EN dataset is {stela_total_token_rejection_rate:.2%}
Type  rejection rate across the STELATranscription/EN dataset is {stela_total_type_rejection_rate:.2%}
""")

,set,tokens,types
0,raw,"294,108,863","336,023"
1,rejected,"4,445,425","203,824"
2,clean,"289,663,438","132,199"



Token rejection rate across the STELATranscription/EN dataset is 1.51%
Type  rejection rate across the STELATranscription/EN dataset is 60.66%



#### Section Rejection rates

To calculate word rates in the dataset, we use the method of cutting each split into chunk of a specific size, and then proceed to calculate the rejection rate/acceptance rate. 
We do this to allow verification of the averages and to allow comparisons with CHILDES as the CHILDES & others datasets do not have the same size. 

In [1]:
import platform

import pandas as pd
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella, utils as dataset_utils
from lexical_benchmark.stats import block_average
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


freq_builder = stella.STELAWordFrequencies()
dataset = stella.STELATranscriptDataset()
word_dict = {
    "EN": dataset_utils.DictionairyCleaner(lang="EN"),
}

CHUNK_SIZE = 16_000
%store -r stela_sections_ws_16k
compute_16k = True
try:
    if len(stela_sections_ws_16k) > 0:
        compute_16k = False
except NameError:
    pass

if compute_16k:
    with timed_status(status="Computing cleaning word stats with 16k blocks", complete_status="Succesfully 16k block average !"):
        stela_sections_ws_16k = {}
        for item in dataset.iter_all():
            # Grabbing the pre-processed but not dictionairy validated text
            raw_words = item.processed_raw_transcription.read_tokenized()
            word_chunk_list = block_average.split_and_fill_chunks(raw_words, chunk_size=CHUNK_SIZE)
            stela_sections_ws_16k[item.section_id] = block_average.calculate_block_word_filtering_rates(
                word_chunk_list, dictionairy=word_dict[item.lang]
            )
    %store stela_sections_ws_16k
else:
    print("16k block stats already computed skipping...")

CHUNK_SIZE = 1600
%store -r stela_sections_ws_1k6h
compute_1k6h = True
try:
    if len(stela_sections_ws_1k6h) > 0:
        compute_1k6h = False
except NameError:
    pass

if compute_1k6h:
    with timed_status(status="Computing cleaning word stats with 1.6k blocks", complete_status="Succesfully 1.6k block average !"):
        stela_sections_ws_1k6h = {}
        for item in dataset.iter_all():
            # Grabbing the pre-processed but not dictionairy validated text
            raw_words = item.processed_raw_transcription.read_tokenized()
            word_chunk_list = block_average.split_and_fill_chunks(raw_words, chunk_size=CHUNK_SIZE)
            stela_sections_ws_1k6h[item.section_id] = block_average.calculate_block_word_filtering_rates(
                word_chunk_list, dictionairy=word_dict[item.lang]
            )
    %store stela_sections_ws_1k6h
else:
    print("1.6k block stats already computed skipping...")

16k block stats already computed skipping...
no stored variable or alias stela_sections_ws_1k6h


Output()

Succesfully 1.6k block average ! (Total time: 7 minutes and 28 seconds)

Stored 'stela_sections_ws_1k6h' (dict)


In [3]:
%store -r stela_sections_ws_1k6h
%store -r stela_sections_ws_16k
import pandas as pd
from lexical_benchmark.utils import ipython_utils

view_cfg_types = {"view_type": "result_types", "avg_type": "median"}
view_cfg_tokens = {"view_type": "result_tokens", "avg_type": "median"}

stela_sections_ws_1k6h_df_tokens = [
    {"Section": k, **v.view(**view_cfg_tokens)} for k, v in stela_sections_ws_1k6h.items()
]
stela_sections_ws_1k6h_df_types = [
    {"Section": k, **v.view(**view_cfg_types)} for k, v in stela_sections_ws_1k6h.items()
]

stela_sections_ws_16k_df_tokens = [
    {"Section": k, **v.view(**view_cfg_tokens)} for k, v in stela_sections_ws_16k.items()
]
stela_sections_ws_16k_df_types = [
    {"Section": k, **v.view(**view_cfg_types)} for k, v in stela_sections_ws_16k.items()
]

ipython_utils.display_side_by_side(
    dataframes={
        "Tokens": (
            ("STELA 16k Block Average", pd.DataFrame(stela_sections_ws_16k_df_tokens)),
            ("STELA 1.6k Block Average", pd.DataFrame(stela_sections_ws_1k6h_df_tokens))
        ),
        "Types": (
            ("STELA 16k Block Average", pd.DataFrame(stela_sections_ws_16k_df_types)),
            ("STELA 1.6k Block Average", pd.DataFrame(stela_sections_ws_1k6h_df_types))
        ),
    },
    custom_format={
        'Tokens': '{:,}',
        'Tokens Rejected': '{:,}',
        'Token Rejection': '{:.2%}',
        'Tokens Accepted': '{:,}',
        'Token Acceptance': '{:.2%}',
        'Types': '{:,}',
        'Types Rejected': '{:,}',
        'Type Rejection': '{:.2%}',
        'Types Accepted': '{:,}',
        'Type Acceptance': '{:.2%}'
    },
)

,Section,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
0,EN_50h_00,"480,000","2,991",0.53%,"477,009",99.47%
1,EN_50h_01,"688,000","5,688",0.81%,"682,312",99.19%
2,EN_50h_02,"544,000","7,034",1.35%,"536,966",98.65%
3,EN_50h_03,"560,000","4,254",0.38%,"555,746",99.62%
4,EN_50h_04,"576,000","4,460",0.64%,"571,540",99.36%
5,EN_50h_05,"464,000","5,021",0.97%,"458,979",99.02%
6,EN_50h_06,"448,000","4,472",0.47%,"443,528",99.53%
7,EN_50h_07,"768,000","2,899",0.13%,"765,101",99.87%
8,EN_50h_08,"528,000","6,349",0.48%,"521,651",99.52%
9,EN_50h_09,"416,000","2,239",0.41%,"413,761",99.59%


## Per Book Word Cleaning Rates

We compute word cleaning per book to see if any books are contain more bad words than others.

In [6]:
import platform
import pandas as pd
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.stats import block_average
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


def word_tokenization(txt: list[str]) -> list[str]:
    words = []
    for line in txt:
        words.extend(line.split())
    return words


CHUNK_160k = 160_000
CHUNK_16k = 160_000
dataset = stella.STELATranscriptDataset()
book_index = stella.STELATranscriptionBookIndex()
all_books = []
for item in dataset.iter_all():
    all_books.extend(item.book_names)

# Check cache for pre-computed stats
%store -r stela_book_stats_16k
%store -r stela_book_stats_1k6
compute_books_stats = True
try:
    if len(stela_book_stats_16k) > 0 and len(stela_book_stats_1k6) > 0:
        compute_books_stats = False
except NameError:
    pass

all_books = set(all_books)
clean_rules = dataset.clean_up_rules("EN")
clean_lexicon = dataset_utils.DictionairyCleaner(lang="EN")
loading_status = "Parsing book text & extracting stats"
complete_status = "Completed parsing & extraction of all books !!"


if compute_books_stats:
    with timed_status(status=loading_status, complete_status=complete_status):
        stela_book_stats_16k = {}
        stela_book_stats_1k6 = {}
        for book_id in all_books:
            book_file = book_index.book2path(lang="EN", book=book_id)

            # Remove non-word items from text
            clean_txt = dataset_utils.DatasetCleaner.clean_txt(
                txt_dirty=book_file.safe_readlines(),
                ruleset=clean_rules
            )
            # Split into 16k sized chunks & extract validation stats
            word_chunk_list = block_average.split_and_fill_chunks(word_tokenization(clean_txt), chunk_size=16_000)
            stela_book_stats_16k[book_id] = block_average.calculate_block_word_filtering_rates(
                word_chunk_list, dictionairy=clean_lexicon
            )

            # Split into 1.6k sized chunks & extract validation stats
            word_chunk_list = block_average.split_and_fill_chunks(word_tokenization(clean_txt), chunk_size=1600)
            stela_book_stats_1k6[book_id] = block_average.calculate_block_word_filtering_rates(
                word_chunk_list, dictionairy=clean_lexicon
            )
        %store stela_book_stats_16k
        %store stela_book_stats_1k6
else:
    print('Loaded book stats from cache...')

no stored variable or alias stela_book_stats_16k
no stored variable or alias stela_book_stats_1k6


Output()

Stored 'stela_book_stats_16k' (dict)

Stored 'stela_book_stats_1k6' (dict)

Completed parsing & extraction of all books !! (Total time: 15 minutes and 29 seconds)

In [9]:
%store -r stela_book_stats_16k
%store -r stela_book_stats_1k6
import pandas as pd
from lexical_benchmark.utils import ipython_utils

view_cfg_types = {"view_type": "result_types", "avg_type": "median"}
view_cfg_tokens = {"view_type": "result_tokens", "avg_type": "median"}

stela_book_ws_16k_df_tokens = [
    {"Book": k, **v.view(**view_cfg_tokens)} for k, v in stela_book_stats_16k.items()
]
stela_book_ws_16k_df_types = [
    {"Book": k, **v.view(**view_cfg_types)} for k, v in stela_book_stats_16k.items()
]

stela_book_ws_1k6h_df_tokens = [
    {"Book": k, **v.view(**view_cfg_tokens)} for k, v in stela_book_stats_1k6.items()
]
stela_book_ws_1k6h_df_types = [
    {"Book": k, **v.view(**view_cfg_types)} for k, v in stela_book_stats_1k6.items()
]

ipython_utils.display_side_by_side(
    dataframes={
        "Tokens": (
            ("Stela Source Books(16k Block)", pd.DataFrame(stela_book_ws_16k_df_tokens)),
            ("Stela Source Books(1.6k Block)", pd.DataFrame(stela_book_ws_1k6h_df_tokens))
        ),
        "Types": (
            ("Stela Source Books(16k Block)", pd.DataFrame(stela_book_ws_16k_df_types)),
            ("Stela Source Books(1.6k Block)", pd.DataFrame(stela_book_ws_1k6h_df_types))
        ),
    },
    custom_format={
        'Tokens': lambda x: '{:,}'.format(int(x)),
        'Tokens Rejected': lambda x: '{:,}'.format(int(x)),
        'Token Rejection': '{:.2%}',
        'Tokens Accepted': lambda x: '{:,}'.format(int(x)),
        'Token Acceptance': '{:.2%}',
        'Types': lambda x: '{:,}'.format(int(x)),
        'Types Rejected': lambda x: '{:,}'.format(int(x)),
        'Type Rejection': '{:.2%}',
        'Types Accepted': lambda x: '{:,}'.format(int(x)),
        'Type Acceptance': '{:.2%}'
    },
)

,Book,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
0,6359_LibriVox_en,"16,000",45,0.28%,"15,955",99.72%
1,2749_LibriVox_en,"80,000",163,0.22%,"79,837",99.78%
2,6176_LibriVox_en,"96,000",162,0.13%,"95,838",99.87%
3,1704_LibriVox_en,"32,000",30,0.09%,"31,970",99.91%
4,3516_LibriVox_en,"16,000",16,0.10%,"15,984",99.90%
5,5958_LibriVox_en,0,0,nan%,0,nan%
6,6117_LibriVox_en,"208,000","2,153",1.04%,"205,847",98.96%
7,8822_LibriVox_en,"16,000",232,1.45%,"15,768",98.55%
8,2201_LibriVox_en,0,0,nan%,0,nan%
9,2422_LibriVox_en,"304,000","3,807",1.18%,"300,193",98.83%


In [19]:
from IPython.display import display
from lexical_benchmark.utils import ipython_utils
import pandas as pd

_typedf = pd.DataFrame(stela_book_ws_1k6h_df_types)
_tokendf = pd.DataFrame(stela_book_ws_1k6h_df_tokens)


ipython_utils.display_side_by_side(
    dataframes={
        "Low Books (1.6k Block-AVG)": (
            ("Tokens", _tokendf[_tokendf["Token Acceptance"] < 0.85]),
            ("Types", _typedf[_typedf["Type Acceptance"] < 0.85]),
        )
    },
    custom_format={
        'Tokens': lambda x: '{:,}'.format(int(x)),
        'Tokens Rejected': lambda x: '{:,}'.format(int(x)),
        'Token Rejection': '{:.2%}',
        'Tokens Accepted': lambda x: '{:,}'.format(int(x)),
        'Token Acceptance': '{:.2%}',
        'Types': lambda x: '{:,}'.format(int(x)),
        'Types Rejected': lambda x: '{:,}'.format(int(x)),
        'Type Rejection': '{:.2%}',
        'Types Accepted': lambda x: '{:,}'.format(int(x)),
        'Type Acceptance': '{:.2%}'
    }
)

,Book,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
585,5788_LibriVox_en,"705,600","162,699",22.88%,"542,901",77.12%
640,4955_LibriVox_en,"11,200","1,940",17.81%,"9,260",82.19%
,Book,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
30,4262_LibriVox_en,"51,677","16,223",31.22%,"35,454",68.78%
202,5726_LibriVox_en,"197,998","60,815",30.18%,"137,183",69.82%
418,6897_LibriVox_en,"70,454","11,596",16.38%,"58,858",83.62%
503,6910_LibriVox_en,"25,908","5,889",21.91%,"20,019",78.09%
585,5788_LibriVox_en,"221,386","99,790",44.88%,"121,596",55.12%
640,4955_LibriVox_en,"5,258","1,741",34.65%,"3,517",65.35%


In [20]:
from lexical_benchmark.datasets import stella
bad_books = ["4262_LibriVox_en", "5726_LibriVox_en", "6897_LibriVox_en", "6910_LibriVox_en", "5788_LibriVox_en", "4955_LibriVox_en"]
book_index = stella.STELATranscriptionBookIndex()
bad_books_index = {f"{bk}": book_index.book2path(lang="EN", book=bk) for bk in bad_books}
bad_books_index

{'4262_LibriVox_en': PosixPath('/scratch1/projects/lexical-benchmark/v2/datasets/workdir/source/StelaData/text/EN/LibriVox/on_interpretation_1005_librivox_64kb_mp3_text.txt'),
 '5726_LibriVox_en': PosixPath('/scratch1/projects/lexical-benchmark/v2/datasets/workdir/source/StelaData/text/EN/LibriVox/de_anima_ge_librivox_64kb_mp3_text.txt'),
 '6897_LibriVox_en': PosixPath('/scratch1/projects/lexical-benchmark/v2/datasets/workdir/source/StelaData/text/EN/LibriVox/masterfj_1211_librivox_gh_64kb_mp3_text.txt'),
 '6910_LibriVox_en': PosixPath('/scratch1/projects/lexical-benchmark/v2/datasets/workdir/source/StelaData/text/EN/LibriVox/treatiseofreligion_ma_librivox_64kb_mp3_text.txt'),
 '5788_LibriVox_en': PosixPath('/scratch1/projects/lexical-benchmark/v2/datasets/workdir/source/StelaData/text/EN/LibriVox/ecclesiastes_wycliffe_1109_librivox_64kb_mp3_text.txt'),
 '4955_LibriVox_en': PosixPath('/scratch1/projects/lexical-benchmark/v2/datasets/workdir/source/StelaData/text/EN/LibriVox/jyl_of_brey